# 00 · End-to-end M5 pipeline

Drives `m5.cv` + `m5.evaluation` against the prepared long-frame parquet.
Every cell here is a thin call into the package — equivalent to 
`make cv-stats && make cv-lgbm` plus inline scoring.

**Prereqs:** `make bootstrap && make prep` so `data/processed/long.parquet` exists.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from m5.config import SETTINGS, set_global_seed
from m5.cv import lgbm_cv, stats_cv
from m5.evaluation import compute_components, wrmsse_for_models
from m5.plots import configure_style

configure_style()
set_global_seed()

df = pd.read_parquet(SETTINGS.processed_dir / 'long.parquet')
df.head()

## Statistical baselines (Theta + AutoETS + SeasonalNaive)

In [ ]:
cv_stats = stats_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
components = compute_components(df[df['ds'] < cv_stats['ds'].min()])
stats_scores = wrmsse_for_models(cv_stats[['unique_id', 'ds', 'y']], cv_stats, components)
stats_scores

## LightGBM global model

In [ ]:
cv_lgbm = lgbm_cv(df, h=SETTINGS.horizon, n_windows=SETTINGS.n_windows)
lgbm_scores = wrmsse_for_models(cv_lgbm[['unique_id', 'ds', 'y']], cv_lgbm, components)
lgbm_scores

## Combined leaderboard

Lower is better. For the official 12-level WRMSSE breakdown, run `06_score.ipynb`.

In [ ]:
leaderboard = pd.concat([stats_scores, lgbm_scores]).sort_values()
leaderboard.to_frame('WRMSSE')